# Accessing GEDI data using NASA Harmony API  



Harmony provides access to services that can transform data from NASA's [Earth Observing Systems Data and Information System (EOSDIS)](https://www.earthdata.nasa.gov/eosdis). [`harmony-py`](https://github.com/nasa/harmony-py) Python package is the recommended way of interacting with Harmony service. This notebook shows how to query and access customized GEDI data outputs using [Harmony Trajectory Subsetter](https://www.earthdata.nasa.gov/data/tools/nasa-harmony). This service supports L2 segmented trajectory (not swath) data and offers variable, temporal, bounding box spatial and polygon spatial subsetting. This subsetter also ensures valid segment indices and sizes following transformation.


## Authenticate

In [ ]:
from harmony import BBox, Client, Collection, Request, CapabilitiesRequest
from datetime import datetime
import json
import earthaccess
import geopandas as gp
import os
from IPython.display import JSON
import h5py
import pandas as pd
from shapely.geometry import Point
import hvplot.pandas

import holoviews as hv
import geoviews as gv
import cartopy
hv.extension('bokeh')

os.chdir('../../')

`earthaccess.login()` is used here to access NASA Earthdata Login (EDL) credentials stored in a .netrc file and allows users to type their credentials and persist them to a .netrc file if one does not exist. 

In [ ]:
auth = earthaccess.login(persist=True)
# auth.token

To access data through Harmony service, we need to create a Harmony Client object using either your EDL token or your Earthdata Login credentials. Below, `username` and `password` are directly provided to `Client` function. See [here](https://github.com/nasa/harmony-py/blob/main/examples/intro_tutorial.ipynb) for other options you can create Harmony Client object. 

In [ ]:
harmony_client = Client(auth=(auth.username, auth.password)) 

## Get the GEDI Collections Harmoney Capabilities

we will start by submitting a capabilities request to view the supported transformation services for the GEDI data in Harmony API. Collection capabilities can be requested using collection `short name` (`short_name='GEDI02_B'`) or `concept ID` which is the NASA EOSDIS collection ID provided in the Common Metadata Repository (CMR) metadata (`collection_id='C3974616135-LPCLOUD'`). Supported transformation services and variables (such as variable subsetting, bounding box subsetting, shapefile subsetting, concatenation and reprojection, etc.) have `True` value. Based on the returned capabilities information for GEDI , we can submit a variable subsetting request, bounding box/shapefile subsetting, and temporal subsetting.

| Products | Version | Concept ID | DOI |
|---|---|---|---|
GEDI01_B | Version 003 | C3974616058-LPCLOUD | 10.5067/GEDI/GEDI01_B.003 |
GEDI02_A | Version 003 | C3974616071-LPCLOUD | 10.5067/GEDI/GEDI02_A.003 |
GEDI02_B | Version 003 | C3974616135-LPCLOUD | 10.5067/GEDI/GEDI02_B.003 |

In [ ]:
capabilities = harmony_client.submit(CapabilitiesRequest(collection_id='C3974616135-LPCLOUD'))
print(json.dumps(capabilities, indent=2))

## Create a Harmony Request

There are several parameters that can be used for the Harmony request. These are the most common request parameters:

`collection` (required): You can specify your collection using a concept ID from the Common Metadata Repository (CMR), or the data product's short name.
`spatial`: A spatial constraint on the data, given as either a bounding box or Well-Known Text (WKT).
`shape`: Subsets a supported collection using a shapefile. Pass the path to a GeoJSON file (.json or .geojson), an ESRI Shapefile (.zip or .shz), or a KML file (.kml).
`temporal`: A date and time constraint on the data. 

Please review [the documentation](https://harmony-py.readthedocs.io/en/latest/) and [the introductory Harmony tutorial ](https://github.com/nasa/harmony-py/blob/main/examples/intro_tutorial.ipynb) for more details. 


### GEDI Collection Concept ID

`collection` is a Required parameter. Below, `concept_id` is saved to a variable.  

In [ ]:
print(capabilities['shortName'], ',', capabilities['conceptId'])

concept_id = capabilities['conceptId']


### GEDI Variable Subset

To learn more about the available layers, you can view the GEDI Dictionaries provided in  [GEDI products' DOI Landing pages](https://www.earthdata.nasa.gov/data/catalog?keyword=GEDI). The code below extracts the available variables for the `GEDI_L2B` data to verify user-defined variables.

In [ ]:
cmr_vars = capabilities['variables']
cmr_vars[0:5]

In [ ]:
gedi_var = [var['name'].split('/', 2)[-1] for var in cmr_vars]
gedi_var[0:25]

In [ ]:
subset_L2B = ['lat_lowestmode', 'lon_lowestmode', 'geolocation/degrade_flag', 'digital_elevation_model', 'elev_lowestmode', 'lat_highestreturn', 'lon_highestreturn', 'elev_highestreturn', 'l2b_quality_flag_rel3', 'pai', 'pai_z', 'pavd_z']

subset = []
for s in subset_L2B:
    var = [v for v in gedi_var if v.endswith(f'{s}')]
    var = list(dict.fromkeys(var))
    if len(var) == 1:
        subset.append(var[0])
        
    elif len(var) > 1:
        var = [v for v in var if v.startswith(f'{s}')]
            
        for l in var:
            if l not in subset:
                subset.append(l) 
    else:
        print(f'No variable found for {s}')

subset

Select the subset of your desired beams. For instance, you can only select Full Power beams ('BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011').

In [ ]:
beams = ['BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']  #['BEAM0000', 'BEAM0001', 'BEAM0010', 'BEAM0011', 'BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']


In [ ]:
subset_var = []
for b in beams:
    beam_subset = [f'/{b}/{layer}' for layer in subset]
    [subset_var.append(i) for i in beam_subset]
subset_var

### Spatial Subset

Both `spatial` and` shape` are the query parameters used for spatial subsetting using bounding box or shapefile/GeoJSON respectively. For the Bounding box, the Harmony `Bbox` class accepts spatial coordinates as decimal degrees in the order of west, south, east, and north coordinates (e.g.(-119.205104, 36.012018, -117.907297, 37.054834)). For the spatial subset using a region of interest, the path to a GeoJSON file (.json or .geojson), an ESRI Shapefile (.zip or .shz), or a kml file (.kml) as `shape` param are acceptable inputs. 

In [ ]:
## path to the GeoJSON
roi = BBox(-119.205104, 36.012018, -117.907297, 37.054834)


### Temporal Subset

For temporal subsetting, the data temporal start and end ranges are used as a datetime object. Below, the start and end dates for the query are selected. 

In [ ]:
temporal_range = {'start': datetime(2022, 4, 1), 
                  'stop': datetime(2022, 4, 20)}

Finally, submit the Harmony request and retrieve the request ID.

In [ ]:
request = Request(
    collection = Collection(id=concept_id),
    spatial = roi,    # for bbox
    # shape = roi,   # for the GeoJSON
    temporal = temporal_range,
    variables = subset_var
)

You can check the validity of the request before submitting a Harmony Request.

In [ ]:
request.is_valid()

Finally, submit the request and retrieve list of the processed data URLs once processing is complete. You may optionally show the progress bar below.

In [ ]:
task = harmony_client.submit(request)
print(f'Harmony request ID: {task}')

print(f'Processing your Harmony request:')
task_json = harmony_client.result_json(task, show_progress=True)

Next, download the subset of data. 

In [ ]:
results = harmony_client.download_all(task, directory='data', overwrite=True)

In [ ]:
file_names = [f.result() for f in results]

Function is defined to create a DataFrame from our HDF5 Harmony subset file. Next, `GeoDataFrame` is created for all the downloaded subset file. 

In [ ]:
def h5_to_dataframe(ds, beams, vars):
    """
    This function takes Harmony subset of GEDI hdf5 and returns a DataFrame. 
    """
    #read the dataset
    gedi_ds = h5py.File(ds,'r')
    # see what is the data product 
    product = gedi_ds['METADATA']['DatasetIdentification'].attrs['shortName']
    fileName = gedi_ds['METADATA']['DatasetIdentification'].attrs['fileName']
    date = datetime.strptime(fileName.rsplit('_')[2], '%Y%j%H%M%S').strftime('%Y-%m-%d %H:%M:%S')
    # Create an empty DataFrame for this beam
    df_beam = pd.DataFrame(columns=vars)
    
    for b in beams:
        data_dic = {}
        for v in vars:
            # print(b,v)
            value = gedi_ds[f'{b}/{v}'][()]
            data_dic[v] = value.tolist() 
            
        df_beam = pd.concat([df_beam, pd.DataFrame(data_dic)],join="inner")
        
        # add product, beam, file name, and date columns 
        df_beam.insert(0, 'product', product)
        df_beam.insert(1, 'Beam', b)
        df_beam.insert(2, 'fileName' , fileName)
        df_beam.insert(3, 'date', date)

    return(df_beam.reset_index(drop=True))


In [ ]:
l2b_df = pd.DataFrame()

for file in file_names:
    print(file)
    gedi_subset = h5_to_dataframe(file, beams, subset)
    l2b_df = gp.GeoDataFrame(pd.concat([l2b_df, gedi_subset]))
    del gedi_subset

# Reset the indeces
l2b_df = l2b_df.reset_index(drop=True)
l2b_df = l2b_df.rename(columns={'lat_lowestmode': 'lat', 'lon_lowestmode': 'lon'})
 # Take the lat/lon from DataFrame and convert each lat/lon to a shapely point and convert to a GeoDataFrame
l2b_df = gp.GeoDataFrame(l2b_df, geometry=l2b_df.apply(lambda row: Point(row.lon, row.lat), axis=1))


In [ ]:
l2b_df.head()

In [ ]:
l2b_df.hvplot.points(
    x='lon', y='lat', c='pai', cmap='viridis',
    clim=(0, l2b_df['pai'].max()), geo=True, tiles='EsriImagery',
    frame_width=900, rasterize=True, dynspread=True,  max_px=15, threshold=0.9,
    colorbar=True, clabel='pai',
)

## Contact Info:  

Email: LPDAAC@usgs.gov  
Voice: +1-866-573-3222  
Organization: Land Processes Distributed Active Archive Center (LP DAAC)¹  
Website: <https://www.earthdata.nasa.gov/>  
¹Work performed under USGS contract G15PD00467 for NASA contract NNG14HH33I.  